# T1-11 Prove the prompt fails, then inspect what extra practice would teach

**Time:** 09:00–10:30 (architecture 09:00–09:15) &nbsp;|&nbsp; **Evidence:** pre-flight JSON in `logs/` + training-row report

**Notebook learning outcomes**

- Shade the architecture card: **frozen** base vs a small **adapter**.
- Split **burst** training from **standing** inference.
- **Prove** that a careful prompt still fails on this machine (or say clearly that it did not).
- Inspect the **criteria** training rows the adapter would see. `human_review` is the centre of the job.


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


**Data ban:** CivicDesk fiction only. The frozen held-out cases are a **test** set. Do not train on them. Do not paste ministry CVs, tickets or minutes.


## Overnight readers — two pictures before any rows

Day 1 architecture card (keep it):

```
input text → tokenizer → embedding → transformer blocks → LM head → next token
```

**Training** updates weights. **Inference** uses frozen weights. Yesterday RAG did **not** update weights.

**Fine-tuning** here is a **LoRA adapter**: a small extra file on some of the blocks. The base stays frozen. Disable the file = rollback.

**Burst vs standing**

| | Burst | Standing |
|---|---|---|
| What | A training (or LoRA) **job** | The chat **API** that stays up |
| When | Hours, then stop the extra GPU | All day on `:8000` / `:8001` |
| Fail | Buying one box “for AI” with no split | Assuming a LoRA replaces the API |

**Headline task today:** résumé **criteria** (`meets` / `does_not_meet` / `human_review`). Tickets are Track A. Redaction stays aligned with the Day 1 40-document test. Minutes are a stretch.

**The prompt is a description of the job.** One worked example in a prompt is copied. Hundreds of labelled rows in training teach the pattern.


## Ministry link

| Cluster | What fails in the frozen set |
|---|---|
| **Labour / MOLSED** | Criteria: silence labelled as failure; in-progress counted as meets |
| **Trade / MTIT** | Tickets: security buried under a printer; none of four incidents escalated in the CPU reference |
| **Rural Development / MRDLG** | Minutes: a defeated motion logged as work to chase |
| **all PII papers** | Redaction: example header pasted, then the real name underneath |

This lab is **N1** (quote a span) and **N2** (human_review is not a fail).


## 0. Backend


In [ ]:
# [CHANGE] olmo is the classroom default. phi needs :8001. demo = labelled stand-ins, not scored.
BACKEND = "olmo"  # "olmo" | "phi" | "demo"

import civicdesk_ft as ft

cfg = ft.set_backend(BACKEND)


## 1. Where a LoRA would sit

`print_anatomy` reads the OLMo **config**, not the weights. Type the attention names from the printout.


In [ ]:
# [RUN]
import civicdesk_ft as ft

ft.print_anatomy()


In [ ]:
# [CHANGE] fill from the printout. Then try four attention names, then all seven.
TARGETS = ["q_proj", "v_proj"]
RANK = 8

import civicdesk_ft as ft

ft.lora_param_count(TARGETS, RANK)


## 2. Pre-flight — prove the problem on *this* machine

Load the frozen 20-case set (7 CVs, 9 tickets, 4 excerpts). Call the **base** model at temperature 0. Save a dated JSON.

The notebook **must not** reuse percentages from a briefing note. If a tripwire does not fire, that is a real result: print the notice. Do not fake a failure.


In [ ]:
# [CHANGE] drop a task if the room is short on time. Default is all four.
TASKS = ["criteria", "tickets", "redaction", "minutes"]

import civicdesk_ft as ft

record = ft.run_preflight(TASKS)


### Teaching note

- **criteria** is scored **one criterion per call** — the format the adapter will train on.
- R2 is also stored as a **multi-criteria** call so the teaching card matches the prompt that copied the example.
- `BACKEND=demo` or a down server: the cell writes an unmeasured file and tells you to use the labelled CPU cards. Those cards are **not** this session's score.


## 3. Three failures, one per scenario


In [ ]:
# [CHANGE] "R2-criteria" | "T1" | "M1"
SHOW = "R2-criteria"

import civicdesk_ft as ft

ft.show_failure(SHOW)


**Write-up:** in one line, what did the prompt *ask* for, and what did the model *do*? Then change `SHOW` and run the other two.


## 4. Generate the criteria training rows

Seeded script. Asserts: evidence is a substring, `does_not_meet` never has empty evidence, no held-out names or criterion types.


In [ ]:
# [RUN]
import runpy
from pathlib import Path

runpy.run_path(str(Path("build_criteria_set.py")), run_name="__main__")


In [ ]:
# [RUN]
import civicdesk_ft as ft

rows = ft.inspect_criteria_rows()


**Write-up:** which of your T1-05 gold pairs does this set cover, and which does it miss? Silence → `human_review` is the policy this file teaches. A stricter unit might label an unrelated degree as `does_not_meet`. Say which policy you would take back.


## 5. Also inspect the redaction hunt (same 40-document test as Day 1)

The redaction adapter, if it exists, is scored on **that** test, not on today's 7-CV stress slice. Find five planted problems in the inspect file.


In [ ]:
# [RUN]
import civicdesk_ft as ft

ft.inspect_training_rows()


In [ ]:
# [CHANGE] put the row id, or None for the set-level problem.
FOUND = {"leak": None, "inconsistent": None, "canary": None, "rewrite": None, "imbalance": None}

import civicdesk_ft as ft

ft.score_hunt(FOUND)


## Appendix — CivicDesk warm-up rows

FT-01..FT-10 stay as literacy rows. FT-07 is a near-copy of FT-01: the same leak in miniature. Canaries FT-08 / FT-09 and TR-P3 never enter a real train split.


In [ ]:
# [OPTIONAL] warm-up list
import civicdesk_ft as ft

ft.print_rows()


In [ ]:
# [OPTIONAL] short LoRA job on the learner GPU. Skip with no GPU.
# The evaluated classroom adapter is the pre-built directory, not this run.
!python train_adapter.py --task criteria --max-steps 30 --time-cap-s 180 --out adapters/learner_run
